# Dataset fit review for IdeHistorieCase

Reviewed on 2026-10-30. This notebook compares the two supplied ZIPs with the checked-in Versailles 1919 scenario. Archive documents were treated as data, not instructions; no bundled scripts or tests were executed.

## Decision

Neither archive is ready to become evidence for the current Versailles scenario. The v3.1 corpus is source-derived and technically traceable, but focuses on Danish correspondence from 1523–1937 (only six letters dated 1919) and lacks human OCR/fact verification. The v2.0 package is explicitly a synthetic, provisional retrieval benchmark, so use it for retrieval-pipeline regression only. The game repo currently has a fake turn provider; source references are planned keys and all scenario source access is marked citation-ineligible.

Source paths below point to the two files supplied in Downloads. Change them if you move the archives.


In [ ]:
from pathlib import Path
import collections, hashlib, json, zipfile

COMPLETE_ZIP = Path(r'C:\Users\marcu\Downloads\danish_history_complete_bundle_v3_1 (3).zip')
CANONICAL_ZIP = Path(r'C:\Users\marcu\Downloads\danish_history_canonical_v2_0 (2).zip')

def read_jsonl(zf, member):
    return [json.loads(line) for line in zf.read(member).decode('utf-8-sig').splitlines() if line.strip()]

def sha256(data):
    return hashlib.sha256(data).hexdigest()


## Package integrity

The complete bundle manifest lists 36 payload files with valid sizes and SHA-256 hashes. The v2 archive carries SHA256SUMS for its payload, and the complete bundle embeds the same v2 ZIP bytes as the separately supplied archive.


In [ ]:
with zipfile.ZipFile(COMPLETE_ZIP) as zf:
    base = 'danish_history_complete_bundle_v3_1/'
    manifest = json.loads(zf.read(base + 'BUNDLE_MANIFEST.json'))
    bundle_checks = []
    for item in manifest['files']:
        payload = zf.read(base + item['path'])
        bundle_checks.append(len(payload) == item['size_bytes'] and sha256(payload) == item['sha256'])
    embedded_v2 = zf.read(base + 'previous_releases/danish_history_canonical_v2_0.zip')

with zipfile.ZipFile(CANONICAL_ZIP) as zf:
    sums_path = 'danish_history_canonical_v2/SHA256SUMS.txt'
    checksum_checks = []
    for row in zf.read(sums_path).decode('utf-8-sig').splitlines():
        parts = row.split(None, 1)
        if len(parts) == 2:
            expected, name = parts
            payload = zf.read('danish_history_canonical_v2/' + name.strip().lstrip('*'))
            checksum_checks.append(sha256(payload) == expected)

print({
    'complete_manifest_entries': len(bundle_checks),
    'complete_manifest_hashes_ok': sum(bundle_checks),
    'v2_checksum_entries': len(checksum_checks),
    'v2_checksum_hashes_ok': sum(checksum_checks),
    'embedded_v2_equals_separate_zip': embedded_v2 == CANONICAL_ZIP.read_bytes(),
})


## v3.1 authentic corpus profile

Grain: one letter per document and source-exact overlapping passages (120-word target, 20-word overlap). Hashes and character spans establish internal consistency and correspondence to the bundled TEI editions; they do not establish OCR accuracy against printed pages or historical truth.


In [ ]:
with zipfile.ZipFile(COMPLETE_ZIP) as zf:
    data_root = 'danish_history_complete_bundle_v3_1/latest/authentic_corpus_v3_1/data/authentic/'
    docs = read_jsonl(zf, data_root + 'documents.jsonl')
    passages = read_jsonl(zf, data_root + 'passages.jsonl')

doc_by_id = {doc['id']: doc for doc in docs}
years = collections.Counter(doc.get('year') for doc in docs if isinstance(doc.get('year'), int))
span_matches = passage_hash_matches = 0
for passage in passages:
    parent = doc_by_id.get(passage['document_id'])
    if parent and parent['original_text'][passage['char_start']:passage['char_end']] == passage['contents']:
        span_matches += 1
    if sha256(passage['contents'].encode('utf-8')) == passage['text_sha256']:
        passage_hash_matches += 1
document_hash_matches = sum(sha256(doc['original_text'].encode('utf-8')) == doc['text_sha256'] for doc in docs)

print({
    'letters': len(docs), 'passages': len(passages),
    'unique_letter_ids': len(doc_by_id),
    'unique_passage_ids': len({p['id'] for p in passages}),
    'orphan_passages': sum(p['document_id'] not in doc_by_id for p in passages),
    'exact_passage_spans': span_matches,
    'valid_passage_text_hashes': passage_hash_matches,
    'valid_letter_text_hashes': document_hash_matches,
    'letters_dated_1919': years.get(1919, 0),
    'letters_dated_1914_to_1920': sum(n for y, n in years.items() if 1914 <= y <= 1920),
    'year_range': (min(years), max(years)),
    'undetermined_unverified_language': sum(d.get('language') == 'und' and not d.get('language_verified') for d in docs),
    'ocr_accuracy_checked': sum(bool(d.get('ocr_accuracy_checked')) for d in docs),
    'historical_accuracy_checked': sum(bool(d.get('factual_accuracy_checked')) for d in docs),
})


## v2.0 retrieval benchmark profile

The benchmark has 300 queries, 150 authored passages, 51 source references, and 3,600 provisional judgments. Its 10 Versailles queries are all assigned to test, but the passages are not archival evidence and the relevance labels are not independently reviewed.


In [ ]:
with zipfile.ZipFile(CANONICAL_ZIP) as zf:
    root = 'danish_history_canonical_v2/'
    corpus = read_jsonl(zf, root + 'exports/corpus.jsonl')
    queries = read_jsonl(zf, root + 'exports/queries.jsonl')
    judgments = read_jsonl(zf, root + 'exports/judgments.jsonl')
    sources = read_jsonl(zf, root + 'exports/sources.jsonl')
    eval_report = json.loads(zf.read(root + 'reports/bm25_test_eval.json'))

print({
    'queries': len(queries), 'passages': len(corpus), 'sources': len(sources), 'judgments': len(judgments),
    'passage_types': dict(collections.Counter(p['metadata'].get('text_type') for p in corpus)),
    'judgments_marked_for_human_review': sum(j.get('requires_human_review') is True for j in judgments),
    'independently_double_annotated': sum(j.get('independently_double_annotated') is True for j in judgments),
    'historical_source_checked_judgments': sum(j.get('historical_source_checked') is True for j in judgments),
    'sources_with_verified_urls': sum(s.get('url_verified_in_this_release') is True for s in sources),
    'versailles_query_count': sum(q.get('metadata', {}).get('topic_family') == 'versailles1919' for q in queries),
    'versailles_splits': dict(collections.Counter(q['metadata'].get('split') for q in queries if q.get('metadata', {}).get('topic_family') == 'versailles1919')),
    'reported_test_baseline': eval_report['overall'],
    'test_topic_families': eval_report['topic_families'],
})


## Fit against the checked-in game

| Game need | v3.1 authentic corpus | v2.0 canonical benchmark |
|---|---|---|
| Primary evidence for Versailles 1919 | Poor fit: Danish letters, only 6 dated 1919; no verified relevance to the conference | Not evidence: authored summaries and source locators, not archival excerpts |
| Traceable quotations | Good technical basis: per-letter IDs, source URL/metadata, exact passage spans and hashes; letter URLs resolve to shared edition pages | No historical source text to quote |
| OCR/factual verification | Not done against printed originals; no independent claim audit | Most passage claims unverified; all judgments await human review |
| Retrieval QA | Not a gold benchmark; no independent qrels | Good software smoke/regression set only; fixed pools and provisional labels |
| Direct game integration | Not supported by current scenario bundle contract; corpus is separate JSONL/SQLite/Postgres | Not supported; separate benchmark exports |

Game basis: `README.md` says the app is an M2 prototype with a deterministic fake provider, not a complete playable game; `docs/SCENARIO_AUTHORING.md` says `source_refs` are planned document keys, not ingested sources or valid citations; the Versailles evidence sidecar marks every source `citation_ready: false`; `backend/app/scenarios/bundle.py` includes only the manifest, versioned YAML sidecars, and YAML files under `sources/` in the scenario hash.

## Recommended next step

For Versailles, ingest and review the actual 1919 sources already cataloged for this scenario, preserving exact text spans, source IDs, document dates, actor access, rights state, and in-world availability dates. Keep v2.0 isolated as retrieval regression data. Keep v3.1 for a separate Danish-history scenario after a human OCR sample and source/date/metadata review.

## Limits

This was a static review of the supplied archives and checked-in scenario files. It did not execute archive scripts, run the game, or test retrieval quality against either dataset. No historical claim was independently adjudicated here.
